# 00 · Extracción de la tienda 44

Este notebook extrae de `train.csv` (Corporación Favorita, nivel SKU) las ventas de la **tienda 44** en el período **2016‑01‑02 a 2017‑08‑15** y les agrega los atributos de cada SKU (`items.csv`).

| | |
|---|---|
| **Entrada** | `../data/train.csv`, `../data/items.csv` (archivos originales de Kaggle; no se modifican) |
| **Salida** | `../data/procesados/t44_registros.parquet` |
| **Decisiones que implementa** | D‑01 (datos a nivel SKU), D‑04 (tienda 44), D‑05 (período) |

**Por qué DuckDB.** `train.csv` pesa 5 GB y tiene 125 millones de filas. DuckDB lo filtra con una consulta SQL sin cargarlo en memoria, en segundos. Reemplaza a los notebooks `train_2016_2017`, `union_bbdd` y `filtro_tienda44` (ahora en `GitHub Code/legacy/`), que recorrían el archivo por partes.

**Qué no se une aquí.** Las tablas de tiendas, transacciones y feriados se incorporan en `02_calendario`, por día y para la tienda 44. El precio del petróleo no se usa (D‑11).

In [1]:
# ------------------------------------------------------------------------------------------------------------
# Este notebook extrae las ventas de la tienda 44 en el período de estudio desde train.csv.
# ------------------------------------------------------------------------------------------------------------
# Implementación: Ramiro Saltos
# Fecha de versión: 2026-10-15
# Revisado: 2026-10-15
# ------------------------------------------------------------------------------------------------------------

# Librerías
# ------------------------------------------------------------------------------------------------------------
import os
import time

%pip install duckdb
import duckdb
%pip install pandas
import pandas as pd

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## 1. Parámetros

Todos los parámetros del notebook están en esta celda. Las rutas son relativas a la carpeta de los notebooks: los datos originales están en `../data/` y los archivos que genera el pipeline, en `../data/procesados/`.

In [2]:
# Parámetros del diseño acordado
# ------------------------------------------------------------------------------------------------------------
tienda = 44                      # D-04
fecha_inicio = "2016-01-02"      # D-05
fecha_fin = "2017-08-15"         # D-05

# Rutas
# ------------------------------------------------------------------------------------------------------------
ruta_datos = os.path.join("..", "data")
ruta_procesados = os.path.join(ruta_datos, "procesados")
archivo_train = os.path.join(ruta_datos, "train.csv")
archivo_items = os.path.join(ruta_datos, "items.csv")
archivo_salida = os.path.join(ruta_procesados, "t44_registros.parquet")

# Valores esperados para el diseño acordado (verificados el 2026-10-15); si cambian los parámetros, actualizarlos
filas_esperadas = 1_599_702
skus_esperados = 3_656
dias_esperados = 590

os.makedirs(ruta_procesados, exist_ok=True)

## 2. Extracción con DuckDB

La consulta lee `train.csv` con tipos declarados, filtra tienda y período, y une los atributos del SKU. La tabla resultante tiene una fila por SKU‑día **con registro**: recuerda que `train.csv` no trae filas para los días sin venta (M‑03). El panel completo con ceros se construye en `01_panel_sku`.

In [3]:
def extraer_tienda(
    archivo_train: str,
    archivo_items: str,
    tienda: int,
    fecha_inicio: str,
    fecha_fin: str,
) -> pd.DataFrame:
    """
    Esta función extrae de train.csv los registros de una tienda en un período y les agrega los atributos del SKU.

    :param archivo_train:   ruta a train.csv (nivel tienda x SKU x día).
    :param archivo_items:   ruta a items.csv (atributos de cada SKU).
    :param tienda:          número de la tienda (store_nbr).
    :param fecha_inicio:    primera fecha del período, en formato AAAA-MM-DD.
    :param fecha_fin:       última fecha del período, en formato AAAA-MM-DD.
    :return:                un DataFrame con una fila por SKU-día con registro y las columnas date, item_nbr,
                            unit_sales, onpromotion, family, class y perishable.
    """

    # Usar barras normales en las rutas para que la consulta funcione también en Windows
    ruta_train = archivo_train.replace("\\", "/")
    ruta_items = archivo_items.replace("\\", "/")

    consulta = f"""
        SELECT t.date, t.item_nbr, t.unit_sales, t.onpromotion, i.family, i.class, i.perishable
        FROM read_csv('{ruta_train}', header = true,
                      columns = {{'id': 'BIGINT', 'date': 'DATE', 'store_nbr': 'SMALLINT', 'item_nbr': 'INTEGER',
                                  'unit_sales': 'DOUBLE', 'onpromotion': 'BOOLEAN'}}) AS t
        JOIN read_csv('{ruta_items}', header = true) AS i USING (item_nbr)
        WHERE t.store_nbr = {tienda}
          AND t.date BETWEEN DATE '{fecha_inicio}' AND DATE '{fecha_fin}'
        ORDER BY t.item_nbr, t.date
    """

    # Ejecutar la consulta y traer el resultado a pandas
    con = duckdb.connect()
    registros = con.sql(consulta).df()
    con.close()

    return registros


inicio = time.time()
registros = extraer_tienda(archivo_train, archivo_items, tienda, fecha_inicio, fecha_fin)
print(f"Extracción terminada en {time.time() - inicio:.1f} segundos")

Extracción terminada en 26.8 segundos


## 3. Tipos de datos y validaciones

Se declaran los tipos para que el archivo de salida ocupe poco espacio y se lea igual en todos los equipos. Las validaciones detienen el notebook si algo no coincide con lo esperado.

In [4]:
# Declarar los tipos de cada columna
registros = registros.astype(
    {
        "item_nbr":     "int32",
        "unit_sales":   "float32",
        "onpromotion":  "boolean",
        "family":       "category",
        "class":        "int16",
        "perishable":   "bool",
    }
)
registros["date"] = pd.to_datetime(registros["date"]).astype("datetime64[ns]")

# Verificar la granularidad, la cobertura y los valores esperados
n_filas = len(registros)
n_skus = registros["item_nbr"].nunique()
n_dias = registros["date"].nunique()

assert not registros.duplicated(["item_nbr", "date"]).any(), "hay más de una fila por SKU-día"
assert registros["onpromotion"].notna().all(), "onpromotion tiene nulos en el período (no debería, ver D-05)"
assert registros["date"].min() >= pd.Timestamp(fecha_inicio), "hay fechas anteriores al inicio del período"
assert registros["date"].max() <= pd.Timestamp(fecha_fin), "hay fechas posteriores al fin del período"
assert n_filas == filas_esperadas, f"se esperaban {filas_esperadas:,} filas y hay {n_filas:,}"
assert n_skus == skus_esperados, f"se esperaban {skus_esperados:,} SKUs y hay {n_skus:,}"
assert n_dias == dias_esperados, f"se esperaban {dias_esperados} días con operación y hay {n_dias}"

print(f"Filas (SKU-días con registro): {n_filas:,}")
print(f"SKUs distintos:                {n_skus:,}")
print(f"Familias:                      {registros['family'].nunique()}")
print(f"Días con operación:            {n_dias}  ({registros['date'].min().date()} a {registros['date'].max().date()})")
print(f"Filas con venta negativa:      {(registros['unit_sales'] < 0).sum():,}  (devoluciones, ver D-20)")
print(registros.dtypes)

Filas (SKU-días con registro): 1,599,702
SKUs distintos:                3,656
Familias:                      32
Días con operación:            590  (2016-01-02 a 2017-08-15)
Filas con venta negativa:      210  (devoluciones, ver D-20)
date           datetime64[ns]
item_nbr                int32
unit_sales            float32
onpromotion           boolean
family               category
class                   int16
perishable               bool
dtype: object


## 4. Exportación

In [5]:
# Guardar en Parquet: 1,6 millones de filas no caben en Excel y Parquet conserva los tipos de datos (A-07)
os.makedirs(os.path.dirname(archivo_salida), exist_ok=True)
registros.to_parquet(archivo_salida, index=False)

print(f"Guardado: {archivo_salida}  ({os.path.getsize(archivo_salida) / 1024**2:.1f} MB)")

print(f"\n>>>> Proceso finalizado <<<<")

Guardado: ..\data\procesados\t44_registros.parquet  (3.4 MB)

>>>> Proceso finalizado <<<<
